## Imports

In [1]:
from pathlib import Path
import sys
import os

In [2]:
import numpy as np

import OpenGL.GL as gl
import OpenGL.GLUT as glut
import pycuda.gl as cudagl

import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

In [3]:
project_working_dir = str(Path(sys.path[0]).parent)
sys.path += [project_working_dir]
os.chdir(project_working_dir)

In [4]:
from src.utils.file_io import read_file_str, show_formatted_cpp

In [5]:
!cl

usage: cl [ option... ] filename... [ /link linkoption... ]


Microsoft (R) C/C++ Optimizing Compiler Version 19.43.34810 for x64
Copyright (C) Microsoft Corporation.  All rights reserved.



## Global parameters

In [6]:
NR_VERTICES = 6
ELEMENTS_PER_OPEN_GL_LINE = 8
BYTES_PER_FLOAT = 4
DIMENSIONS_PER_VERTEX = 3
BLOCK_SIZE = 1024
NR_BLOCKS = (NR_VERTICES + BLOCK_SIZE - 1) // BLOCK_SIZE

### Helper functions

In order to check the output has been written to vertex data in OpenGL, we use this function.

In [7]:
def read_vbo(vbo: int):
    """Read back VBO into numpy array (num_vertices × 8 floats)."""
    gl.glBindBuffer(gl.GL_ARRAY_BUFFER, vbo)
    data = gl.glGetBufferSubData(
        gl.GL_ARRAY_BUFFER, 0, NR_VERTICES * ELEMENTS_PER_OPEN_GL_LINE * BYTES_PER_FLOAT
    )
    gl.glBindBuffer(gl.GL_ARRAY_BUFFER, 0)
    return np.frombuffer(data, dtype=np.float32).reshape(-1, ELEMENTS_PER_OPEN_GL_LINE)

### Create an OpenGL context

In [8]:
glut.glutInit()
glut.glutInitDisplayMode(glut.GLUT_RGBA | glut.GLUT_DOUBLE | glut.GLUT_DEPTH)
glut.glutInitWindowSize(100, 100)
glut.glutCreateWindow(b"PyCUDA-OpenGL Kernel Test")

1

### Create a vertex buffer

In [9]:
vbo = gl.glGenBuffers(1)
gl.glBindBuffer(gl.GL_ARRAY_BUFFER, vbo)
gl.glBufferData(
    gl.GL_ARRAY_BUFFER,
    NR_VERTICES * ELEMENTS_PER_OPEN_GL_LINE * BYTES_PER_FLOAT,
    None,
    gl.GL_DYNAMIC_DRAW,
)
gl.glBindBuffer(gl.GL_ARRAY_BUFFER, 0)

### Register Buffer with cuda

In [10]:
cuda_res = cudagl.RegisteredBuffer(int(vbo), cudagl.graphics_map_flags.WRITE_DISCARD)

### Create Cuda allocated memory

In [11]:
src_vertices = np.arange(NR_VERTICES * DIMENSIONS_PER_VERTEX, dtype=np.float32).reshape(
    NR_VERTICES, DIMENSIONS_PER_VERTEX
)
src_normals = np.arange(
    100, 100 + NR_VERTICES * DIMENSIONS_PER_VERTEX, dtype=np.float32
).reshape(NR_VERTICES, DIMENSIONS_PER_VERTEX)  # distinct values

vertices_gpu = cuda.mem_alloc(src_vertices.nbytes)
cuda.memcpy_htod(vertices_gpu, src_vertices)

normals_gpu = cuda.mem_alloc(src_normals.nbytes)
cuda.memcpy_htod(normals_gpu, src_normals)

In [12]:
src_vertices

array([[ 0.,  1.,  2.],
       [ 3.,  4.,  5.],
       [ 6.,  7.,  8.],
       [ 9., 10., 11.],
       [12., 13., 14.],
       [15., 16., 17.]], dtype=float32)

In [13]:
src_normals

array([[100., 101., 102.],
       [103., 104., 105.],
       [106., 107., 108.],
       [109., 110., 111.],
       [112., 113., 114.],
       [115., 116., 117.]], dtype=float32)

### Compile cuda kernel

In [14]:
cuda_code = read_file_str("./src/cuda_kernels/copy_to_vertex_data.cu")

In [15]:
show_formatted_cpp(cuda_code)

In [16]:
mod = SourceModule(cuda_code)

### Set-up memory for running kernel

In [17]:
copy_to_vertex_data = mod.get_function("copy_to_vertex_data")

In [18]:
nr_vertices = np.uint32(len(src_vertices))

## Profile function

In [19]:
class DeviceAllocationAdapter(cuda.PointerHolderBase):
    def __init__(self, ptr):
        self.gpudata = ptr

    def __int__(self):
        return self.gpudata

In [20]:
def apply_copy_to_vertex_data():
    mapping = cuda_res.map()
    ptr, _ = mapping.device_ptr_and_size()
    try:
        copy_to_vertex_data(
            DeviceAllocationAdapter(ptr),
            vertices_gpu,
            normals_gpu,
            nr_vertices,
            block=(BLOCK_SIZE, 1, 1),
            grid=(NR_BLOCKS, 1, 1),
        )
    finally:
        mapping.unmap()

In [24]:
%timeit apply_copy_to_vertex_data()

208 µs ± 7.34 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Check output is as expected

In [27]:
copied_output = read_vbo(vbo)

In [31]:
assert np.all(np.isclose(copied_output[:, :3], src_vertices))

In [32]:
assert np.all(np.isclose(copied_output[:, 5:], src_normals))